# Pretrained Mini-VLA — SmolVLA in LIBERO (Real Robot VLA Checkpoint)

**AI Builders Lab | Google Colab GPU | Hugging Face LeRobot + MuJoCo LIBERO**

This notebook downloads a **real pretrained robotics VLA checkpoint** (`lerobot/smolvla_libero`) and runs it on a **matching robot manipulation simulator**, LIBERO.

**Important:** Unlike the tiny 2D demonstration, this is a full robotics simulation with camera observations, robot state, language task instructions, and continuous robot actions. You do **not** train the model here; you load existing weights and evaluate them.

**Requirements:** Colab Linux, GPU (T4/L4/A100 recommended), internet, substantial disk space, and time for downloads/compilation. MuJoCo + LIBERO dependency installation may be sensitive to Colab's Python version. This notebook is an **integration recipe based on the official LeRobot evaluation interface**, not a runtime-verified Colab execution. If installation fails, see the troubleshooting section.

## 1. What does the pretrained VLA receive?

**Inputs:** 2 robot camera views + 8D robot proprioceptive state + task-language instruction.

**Output:** a 7D continuous robot control action (6D end-effector delta and gripper command).

**Simulator:** LIBERO/robosuite, a MuJoCo-based robot manipulation benchmark.

**Checkpoint:** `lerobot/smolvla_libero`, which is fine-tuned for LIBERO. This is essential: a generic VLA checkpoint is **not automatically compatible** with an arbitrary robot simulator.

The task instruction comes from the selected LIBERO benchmark task. Changing the task index changes the instruction and the scene together; entering an arbitrary free-form instruction without a matching scene is not a supported experiment.

In [ ]:
import os, sys, shutil, subprocess, torch
from pathlib import Path

print("Python:",sys.version)
print("PyTorch:",torch.__version__)
print("CUDA available:",torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))
else: print("WARNING: select Runtime → Change runtime type → GPU before continuing.")

os.environ["MUJOCO_GL"]="egl" if torch.cuda.is_available() else "osmesa"
os.environ["TOKENIZERS_PARALLELISM"]="false"

## 2. Install LeRobot + LIBERO and repair optional TorchAO conflict

Install the official LeRobot dependencies. The pretrained SmolVLA policy does not need TorchAO quantization, but incompatible TorchAO installations can break Diffusers imports (missing `FqnToConfig`). We remove the **optional** TorchAO package after installation, then immediately smoke-test the exact LeRobot evaluation import. This does not change the SmolVLA checkpoint or remove PyTorch.


In [ ]:
from pathlib import Path
import subprocess, sys
if not Path("/content/lerobot").exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/huggingface/lerobot.git","/content/lerobot"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-e","/content/lerobot[smolvla,libero]"],check=True)
print("Installed LeRobot extras. Restart runtime if imports fail.")
# SmolVLA does not require TorchAO quantization. Some Colab images ship an
# incompatible torchao version; diffusers auto-detects it and imports FqnToConfig.
# Removing this optional package avoids that broken quantization import path.
subprocess.run([sys.executable,"-m","pip","uninstall","-y","torchao"],check=True)

# Validate imports BEFORE launching the slow model evaluation.
check=subprocess.run([sys.executable,"-c",
    "import diffusers; from lerobot.scripts.lerobot_eval import main; "
    "print('LeRobot + diffusers imports OK')"],
    text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(check.stdout[-8000:])
if check.returncode:
    raise RuntimeError("Dependency import smoke test failed. See traceback above.")


## 2b. Initialize LIBERO non-interactively and verify GPU

LIBERO's first import asks an interactive dataset-folder question, which crashes in a non-interactive evaluation subprocess. Run the one-time import below with `N` on stdin to accept LIBERO's default location. We also check whether the installed PyTorch build actually supports CUDA; selecting A100 in Colab does not guarantee a GPU-enabled torch wheel.

In [ ]:
import os, subprocess, sys, torch
print("GPU check:", "torch", torch.__version__, "CUDA build", torch.version.cuda,
      "CUDA available", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: PyTorch cannot see CUDA. Before running evaluation, inspect nvidia-smi and reinstall a CUDA-enabled PyTorch wheel if needed.")
    subprocess.run(["nvidia-smi"],check=False)
# First LIBERO import creates ~/.libero/config.yaml and prompts for dataset location.
# Answer 'N' automatically; subsequent imports use the saved config.
setup = subprocess.run([sys.executable,"-c","from libero.libero import benchmark; print('LIBERO initialized')"],
                       input="N\n",text=True,capture_output=True)
print(setup.stdout[-3000:])
if setup.returncode:
    print(setup.stderr[-5000:])
    raise RuntimeError("LIBERO first-run initialization failed; inspect error above.")
print("LIBERO first-run initialization complete.")

## 3. Choose the language-conditioned manipulation task

These settings are **real simulator task selectors**, not a pretend text-to-action demonstration. LIBERO supplies the scene and its corresponding language goal.

Try `libero_object`, `libero_spatial`, or `libero_goal`. Each has 10 tasks. `TASK_ID` ranges from 0 to 9.

To run a different language instruction, select a different benchmark task. The simulator and VLA both receive the appropriate task context.

In [ ]:
POLICY_ID="lerobot/smolvla_libero"
SUITE="libero_object"
TASK_ID=0
EPISODES=1
OUTPUT_DIR="/content/tiny_vla_libero_eval"
assert SUITE in {"libero_object","libero_spatial","libero_goal","libero_10"}
assert 0<=TASK_ID<=9
print("Checkpoint:",POLICY_ID)
print("Benchmark:",SUITE,"task index:",TASK_ID)

## 4. Run the actual pretrained VLA in the robot simulator

The official `lerobot-eval` command:
1. Downloads the pretrained model checkpoint from Hugging Face.
2. Initializes a matching LIBERO robot manipulation environment.
3. Receives camera images, robot state and the benchmark's language task.
4. Predicts actions with the pretrained VLA and executes them in simulation.
5. Writes evaluation outputs, including videos when supported.

**This can take substantial time** for model download and simulation. Do not interpret a failed install or missing video as a successful robot run.

In [ ]:
import os, subprocess
os.environ["MUJOCO_GL"]="egl"
cmd=[
    "lerobot-eval",
    f"--policy.path={POLICY_ID}",
    "--policy.device=cuda" if torch.cuda.is_available() else "--policy.device=cpu",
    "--env.type=libero",
    '--rename_map={"observation.images.image":"observation.images.camera1","observation.images.image2":"observation.images.camera2"}',
    f"--env.task={SUITE}",
    f"--env.task_ids=[{TASK_ID}]",
    f"--eval.n_episodes={EPISODES}",
    "--eval.batch_size=1",
    "--eval.use_async_envs=false",
    "--env.max_parallel_tasks=1",
    f"--output_dir={OUTPUT_DIR}",
]
print("Executing:", " ".join(cmd))
result=subprocess.run(cmd,env=os.environ.copy(),text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(result.stdout[-16000:])
if result.returncode!=0:
    raise RuntimeError(f"LeRobot evaluation failed with exit code {result.returncode}. The actual traceback is printed above.")
print("Evaluation completed. Inspect the reported success metrics and saved files.")

## 5. Watch the robot execute the instruction

LeRobot typically saves evaluation videos. This cell searches the output folder and displays the first video it finds. If none is present, inspect the evaluation output: the LeRobot version may have different video settings.

In [ ]:
from IPython.display import Video, display
from pathlib import Path
videos=sorted(Path(OUTPUT_DIR).rglob("*.mp4"))
print("Videos found:",len(videos))
if videos:
    display(Video(str(videos[0]),embed=True,width=640))
else:
    print("No MP4 found. Inspect evaluation output and LeRobot video configuration.")
    for p in list(Path(OUTPUT_DIR).rglob("*"))[:35]: print(p)

## 6. Inspect evaluation results

A successful run does not guarantee the robot completes the task. The benchmark evaluates actual task completion. If the robot fails, that is a **model evaluation result**, not proof that the model was never loaded.

In [ ]:
from pathlib import Path
out=Path(OUTPUT_DIR)
for p in sorted(out.rglob("*.json"))[:15]:
    print("\nRESULT:",p)
    print(p.read_text()[:3000])
print("\nTo change the language-conditioned task, modify SUITE/TASK_ID and rerun evaluation.")

## 7. Discussion and limitations

- How do camera images and robot state complement the language instruction?
- Why does the pretrained VLA require a robot and action space matching its training?
- Compare action prediction with the **world model** notebook: which predicts the next action, and which predicts the next state?
- What does **action chunking** mean?
- Why can't we simply type `pick the red block` and expect any scene to support that task?
- How do the benchmark task description, model conditioning, and MuJoCo dynamics interact?

### Troubleshooting

- **No GPU:** choose Colab GPU runtime. CPU inference may be impractically slow.
- **Installation errors:** Colab's Python/package versions may not match upstream LIBERO. Follow the official LeRobot installation docs; a compatible Linux environment may be required.
- **Checkpoint download failure:** check internet connectivity, Hugging Face availability, disk capacity.
- **Rendering failure:** set `MUJOCO_GL=egl` before simulator initialization; restart runtime.
- **Action/observation key mismatch:** ensure the checkpoint matches LIBERO; don't substitute an arbitrary pretrained VLA.
- **Task fails:** try a different task or more episodes; no success is guaranteed.

### Primary references
- [LeRobot repository](https://github.com/huggingface/lerobot)
- [Official LIBERO evaluation guide](https://github.com/huggingface/lerobot/blob/main/docs/source/libero.mdx)
- [Official SmolVLA documentation](https://huggingface.co/docs/lerobot/en/smolvla)
- [LIBERO-tuned SmolVLA checkpoint](https://huggingface.co/lerobot/smolvla_libero)

**Teaching distinction:** The pretrained VLA is genuinely loaded for inference; the environment's language task is selected by benchmark task ID rather than arbitrary user-defined free text.